# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

In [2]:
import numpy as np
import pandas as pd
import matplotlib as plt
from TP.entrega.alc import svd_reducida, norma, inversa, calculaQR, arroba, traspuesta,producto_interno

## GRUPO: FFPL

# Integrantes
* Luis Arroyo
* integrante
* integrante
* integrante

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

### Ejercicio 2 - Calibración con puntos 3D (a)

In [3]:
def obtenerQ3d(imgPts, objPts) -> np.ndarray:
    """
    Arma la matriz Q del sistema homogeneo Qm = 0 del DLT 3D (ecuación 6).

    Parámetros:
        imgPts: array (n, 2) con las coordenadas (u_i, v_i) de cada punto en la imagen, en píxeles.
        objPts: array (n, 3) con las coordenadas (U_i, V_i, W_i) de cada punto en el mundo real.
                El punto i de imgPts debe corresponder al punto i de objPts.
    Retorna:
        Q: array (2n, 12). Las filas 2i y 2i+1 son las dos ecuaciones del punto i.
    """
    imgPts = np.asarray(imgPts, dtype=float)
    objPts = np.asarray(objPts, dtype=float)
    n = objPts.shape[0]

    if imgPts.shape != (n, 2) or objPts.shape != (n, 3):
        raise ValueError("imgPts debe ser (n, 2) y objPts (n, 3), con el mismo n")

    Q = np.zeros((2 * n, 12))

    for i in range(n):
        # Coordenadas del punto i en la imagen
        ui = imgPts[i, 0]
        vi = imgPts[i, 1]

        # Coordenadas del punto i en el mundo real
        Ui = objPts[i, 0]
        Vi = objPts[i, 1]
        Wi = objPts[i, 2]

        Q[2 * i] = [
            Ui, Vi, Wi, 1,
            0, 0, 0, 0,
            -ui * Ui, -ui * Vi, -ui * Wi, -ui
        ]

        Q[2 * i + 1] = [
            0, 0, 0, 0,
            Ui, Vi, Wi, 1,
            -vi * Ui, -vi * Vi, -vi * Wi, -vi
        ]

    return Q

### Ejercicio 2 - Calibración con puntos 3D (b)


In [4]:
def resolverDLT3D(Q):
    # el vector que minimiza ||Qm||^2 es el autovector asociado al menor valor singular
    # que este es el último autovector de la matriz V
    _, _, V = svd_reducida(Q)
    M = V[:,-1].reshape((3, 4))
    # reshape ((3,4)) toma la matriz [1,2,3,4,5,6,7,8,9,10,11,12]
    # y la convierte en una matriz de 3x4 de la forma [[1,2,3,4],[5,6,7,8],[9,10,11,12]]
    return M

### Ejercicio 2 - Calibración con puntos 3D (c)


Ya encontramos $M$:

$$
M =
\begin{bmatrix}
m_{11} & m_{12} & m_{13} & m_{14} \\
m_{21} & m_{22} & m_{23} & m_{24} \\
m_{31} & m_{32} & m_{33} & m_{34}
\end{bmatrix}
=
\begin{bmatrix}
m_1^T \\
m_2^T \\
m_3^T
\end{bmatrix},
\qquad
m_i =
\begin{bmatrix}
m_{i1} \\ m_{i2} \\ m_{i3} \\ m_{i4}
\end{bmatrix}.
$$

Por ejemplo, la primera fila de $M$ es:

$$
m_1^T =
\begin{bmatrix}
m_{11} & m_{12} & m_{13} & m_{14}
\end{bmatrix}.
$$

Queremos obtener $K$, $R$ y $t$, relacionadas mediante
$M = K[R \mid t]$, donde:

$$
K =
\begin{bmatrix}
\alpha_u & \gamma & u_0 \\
0 & \alpha_v & v_0 \\
0 & 0 & 1
\end{bmatrix},
\qquad
[R \mid t] =
\begin{bmatrix}
r_{11} & r_{12} & r_{13} & t_1 \\
r_{21} & r_{22} & r_{23} & t_2 \\
r_{31} & r_{32} & r_{33} & t_3
\end{bmatrix}.
$$

Al realizar el producto, obtenemos:

$$
M = K[R \mid t] =
\begin{bmatrix}
\alpha_u r_{11} + \gamma r_{21} + u_0 r_{31} &
\alpha_u r_{12} + \gamma r_{22} + u_0 r_{32} &
\alpha_u r_{13} + \gamma r_{23} + u_0 r_{33} &
\alpha_u t_1 + \gamma t_2 + u_0 t_3 \\
\alpha_v r_{21} + v_0 r_{31} &
\alpha_v r_{22} + v_0 r_{32} &
\alpha_v r_{23} + v_0 r_{33} &
\alpha_v t_2 + v_0 t_3 \\
r_{31} & r_{32} & r_{33} & t_3
\end{bmatrix}.
$$

También podemos escribir $R$ en términos de sus columnas:

$$
R =
\begin{bmatrix}
r_1 & r_2 & r_3
\end{bmatrix},
\qquad
r_i =
\begin{bmatrix}
r_{1i} \\
r_{2i} \\
r_{3i}
\end{bmatrix}.
$$

La restricción $R^T R = I$ implica que las columnas
$R_1$, $R_2$ y $R_3$ son ortonormales. Es decir:

$$
\|r_i\|_2 = 1
\qquad \text{para todo } i \in \{1,2,3\},
$$

$$
\langle r_i, r_j \rangle = 0
\qquad \text{para todo } i \neq j.
$$





De la última fila de la igualdad $M = K[R \mid t]$, obtenemos:

$$
\begin{bmatrix}
m_{31} & m_{32} & m_{33} & m_{34}
\end{bmatrix}
=
\begin{bmatrix}
r_{31} & r_{32} & r_{33} & t_3
\end{bmatrix}.
$$

Por lo tanto:

$$
m_{31}=r_{31}, \qquad
m_{32}=r_{32}, \qquad
m_{33}=r_{33}, \qquad
m_{34}=t_3.
$$
pero acá tenemos un problema, nosotros no conocemos las caracteristicas de la matriz  $M$, sabemos que la suma de todos sus terminos tiene norma $2$ por como lo construimos, pero en el enunciado me especifican que $R$ tiene que ser ortonormal y $m_{31}^2 + m_{32}^2 + m_{33}^2$, que es $r_{31}^2 + r_{32}^2 + r_{33}^2$, no necesariamente es $1$, entonces lo que tengo que hacer es multiplicar al vector $M$ por algun escalar, tal que los elementos $m_{31}$, $m_{32}$, $m_{33}$ sumados al cuadrado me den $1$.


ahora, si multiplicamos a $M$ por un escalar, no estamos rompiendo la solución? porque nosotros habiamos encontrado que era el autovector asociado al autovalor mas chico, minimiza la solucion $\|Qx\|_2$, pero multiplicar un vector por un escalar no cambia la dirección es decir sigue siendo el mismo atutovector solución, por lo tanto podemos tomar a $M$ y dividirlo por el factor que haga que los elementos $m_{31}$, $m_{32}$, $m_{33}$ que son $r_{31}$, $r_{32}$, $r_{33}$ al cuadrado sumados de uno:

$$
s=\sqrt{m_{31}^2+m_{32}^2+m_{33}^2},
\qquad
\widetilde M=\frac{M}{s}.
$$

Luego este nuevo valor
$$
\qquad
\widetilde M
$$
si cumple que su norma 2 es 1.

Partimos de:

$$
M = K[R \mid t].
$$

Distribuyendo el producto:

$$
M = [KR \mid Kt].
$$

Dividimos M en dos bloques:

$$
M = [M_1 \mid M_2],
$$

donde:

$$
M_1 =
\begin{pmatrix}
m_{11} & m_{12} & m_{13} \\
m_{21} & m_{22} & m_{23} \\
m_{31} & m_{32} & m_{33}
\end{pmatrix}
= KR,
$$

y:

$$
M_2 =
\begin{pmatrix}
m_{14} \\
m_{24} \\
m_{34}
\end{pmatrix}
= Kt.
$$

La matriz K es triangular superior:

$$
K =
\begin{pmatrix}
\alpha_u & \gamma & u_0 \\
0 & \alpha_v & v_0 \\
0 & 0 & 1
\end{pmatrix}.
$$

Como todos sus elementos diagonales son distintos de cero:

$$
\det(K) = \alpha_u \alpha_v \neq 0.
$$

Por lo tanto, K es inversible.

Además, R es ortogonal:

$$
RR^T = R^TR = I.
$$

Por lo tanto, R es inversible y:

$$
R^{-1} = R^T.
$$

Entonces, M₁ es inversible por ser producto de dos matrices inversibles:

$$
M_1 = KR.
$$

Calculamos su inversa:

$$
M_1^{-1}
= (KR)^{-1}
= R^{-1}K^{-1}
= R^TK^{-1}.
$$

Ahora tenemos que:

$$
M_1^{-1} = R^T K^{-1}.
$$

Como $R^T$ es ortogonal y $K^{-1}$ es triangular superior, podemos usar la factorización QR de $M_1^{-1}$.

Para no confundir el factor triangular con la matriz de rotación $R$, lo llamamos $T$:

$$
M_1^{-1} = QT.
$$

$$
Q = R^T,
\qquad
T = K^{-1}.
$$

Por lo tanto, recuperamos la matriz de rotación trasponiendo $Q$:

$$
R = Q^T.
$$

Y recuperamos la matriz de parámetros intrínsecos invirtiendo $T$:

$$
K = T^{-1}.
$$

por ultimo
Como conocemos $M_2$ y ya obtuvimos $K^{-1}$, podemos calcular el vector de traslación $t$.

Partimos de:

$$
M_2 = Kt.
$$

Multiplicando a izquierda por $K^{-1}$:

$$
K^{-1}M_2 = K^{-1}Kt = t.
$$

Por lo tanto:

$$
t = K^{-1}M_2.
$$

Como $T = K^{-1}$, también podemos escribir:

$$
t = TM_2.
$$

In [1]:
def devolverMatricesCalibracion3D(M):
    M_monio = M / norma(M[2, 0:3])
    M1 = M_monio[0:3,0:3]
    M2 = M_monio[:,3]
    M1Inversa = inversa(M1)
    R_traspuesta,K_inversa = calculaQR(M1Inversa,'GS')
    R = traspuesta(R_traspuesta)
    K = inversa(K_inversa)
    t = arroba(K_inversa,M2)
    return R,t,K

### Ejercicio 2 - Calibración con puntos 3D (d)


Para este punto me piden K con datos especificos, no puedo usar la K que me da la factorizacion QR o quizas puedo hacerle alguan transformacion pero hasta ahora no nos respondieron asi que lo que hago es fuerza bruta despejando los parámetros uno por uno de la igualdad $\widetilde M = K[R \mid t]$, fila por fila.


Definimos a $R$ , como filas:

$$
R =
\begin{bmatrix}
\bar r_1^T \\
\bar r_2^T \\
\bar r_3^T
\end{bmatrix},
\qquad
\bar r_i =
\begin{bmatrix}
r_{i1} \\ r_{i2} \\ r_{i3}
\end{bmatrix}.
$$

Como $R$ es ortogonal, también vale $RR^T = I$, así que las filas también son ortonormales:

$$
\|\bar r_i\|_2 = 1,
\qquad
\langle \bar r_i, \bar r_j \rangle = 0 \quad \text{para } i \neq j.
$$



Del mismo modo, llamamos $\bar m_i$ a los tres primeros elementos de la fila $i$ de $\widetilde M$:

$$
\bar m_i =
\begin{bmatrix}
m_{i1} \\ m_{i2} \\ m_{i3}
\end{bmatrix}.
$$

Mirando el producto $K[R \mid t]$ que desarrollamos en el punto (c), cada fila de $\widetilde M$ queda:

$$
\begin{aligned}
\text{fila 3:}\quad & \bar m_3 = \bar r_3, & m_{34} &= t_3, \\
\text{fila 2:}\quad & \bar m_2 = \alpha_v \bar r_2 + v_0 \bar r_3, & m_{24} &= \alpha_v t_2 + v_0 t_3, \\
\text{fila 1:}\quad & \bar m_1 = \alpha_u \bar r_1 + \gamma \bar r_2 + u_0 \bar r_3, & m_{14} &= \alpha_u t_1 + \gamma t_2 + u_0 t_3.
\end{aligned}
$$

**fila 3**
$$
\bar r_3 = \bar m_3,
\qquad
t_3 = m_{34}.
$$

**Fila 2**

Hacemos producto interno de $\bar m_2$ con $\bar r_3$. :

$$
\langle \bar m_2, \bar r_3 \rangle
= \alpha_v \underbrace{\langle \bar r_2, \bar r_3 \rangle}_{0} + v_0 \underbrace{\langle \bar r_3, \bar r_3 \rangle}_{1}
= v_0.
$$

Ahora escribimos la igualdad $\bar m_2 = \alpha_v \bar r_2 + v_0 \bar r_3$ como tres ecuaciones, una por componente:

$$
\begin{aligned}
m_{21} &= \alpha_v r_{21} + v_0 r_{31} \\
m_{22} &= \alpha_v r_{22} + v_0 r_{32} \\
m_{23} &= \alpha_v r_{23} + v_0 r_{33}
\end{aligned}
$$
entonces
$$
\begin{aligned}
m_{21} - v_0 r_{31} &= \alpha_v r_{21} \\
m_{22} - v_0 r_{32} &= \alpha_v r_{22} \\
m_{23} - v_0 r_{33} &= \alpha_v r_{23}
\end{aligned}
$$

Elevamos al cuadrado :

$$
\begin{aligned}
(m_{21} - v_0 r_{31})^2 &= \alpha_v^2 \, r_{21}^2 \\
(m_{22} - v_0 r_{32})^2 &= \alpha_v^2 \, r_{22}^2 \\
(m_{23} - v_0 r_{33})^2 &= \alpha_v^2 \, r_{23}^2
\end{aligned}
$$

Sumamos las tres ecuaciones y sacamos $\alpha_v^2$ de factor común:

$$
(m_{21} - v_0 r_{31})^2 + (m_{22} - v_0 r_{32})^2 + (m_{23} - v_0 r_{33})^2
= \alpha_v^2 \, (r_{21}^2 + r_{22}^2 + r_{23}^2).
$$

Como la fila $\bar r_2$ tiene norma 1, $r_{21}^2 + r_{22}^2 + r_{23}^2 = 1$, y nos queda:

$$
\alpha_v^2 = (m_{21} - v_0 r_{31})^2 + (m_{22} - v_0 r_{32})^2 + (m_{23} - v_0 r_{33})^2.
$$

El lado derecho es la suma de los cuadrados de las componentes del vector $\bar m_2 - v_0 \bar r_3$. Eso es lo mismo que hacer el producto interno de ese vector por sí mismo, así que lo volvemos a escribir en forma vectorial:

$$
\alpha_v^2 = \langle \bar m_2 - v_0 \bar r_3,\ \bar m_2 - v_0 \bar r_3 \rangle.
$$

Sacamos raíz. Como $\alpha_v > 0$, nos quedamos con la positiva:

$$
\alpha_v = \sqrt{\langle \bar m_2 - v_0 \bar r_3,\ \bar m_2 - v_0 \bar r_3 \rangle} = \|\bar m_2 - v_0 \bar r_3\|_2.
$$


$$
\bar r_2 = \frac{\bar m_2 - v_0 \bar r_3}{\alpha_v}.
$$

Por último, de la cuarta columna, $m_{24} = \alpha_v t_2 + v_0 t_3$, despejamos:

$$
t_2 = \frac{m_{24} - v_0 t_3}{\alpha_v}.
$$

**Fila 1**

Hacemos lo mismo, pero ahora hay dos direcciones conocidas, $\bar r_2$ y $\bar r_3$:

$$
u_0 = \langle \bar m_1, \bar r_3 \rangle,
\qquad
\gamma = \langle \bar m_1, \bar r_2 \rangle.
$$

Ahora escribimos la igualdad $\bar m_1 = \alpha_u \bar r_1 + \gamma \bar r_2 + u_0 \bar r_3$ como tres ecuaciones, una por componente:

$$
\begin{aligned}
m_{11} &= \alpha_u r_{11} + \gamma r_{21} + u_0 r_{31} \\
m_{12} &= \alpha_u r_{12} + \gamma r_{22} + u_0 r_{32} \\
m_{13} &= \alpha_u r_{13} + \gamma r_{23} + u_0 r_{33}
\end{aligned}
$$

Ya conocemos $\gamma$, $u_0$ y las filas $\bar r_2$ y $\bar r_3$, así que los pasamos restando:

$$
\begin{aligned}
m_{11} - \gamma r_{21} - u_0 r_{31} &= \alpha_u r_{11} \\
m_{12} - \gamma r_{22} - u_0 r_{32} &= \alpha_u r_{12} \\
m_{13} - \gamma r_{23} - u_0 r_{33} &= \alpha_u r_{13}
\end{aligned}
$$

Elevamos al cuadrado ambos lados de cada ecuación:

$$
\begin{aligned}
(m_{11} - \gamma r_{21} - u_0 r_{31})^2 &= \alpha_u^2 \, r_{11}^2 \\
(m_{12} - \gamma r_{22} - u_0 r_{32})^2 &= \alpha_u^2 \, r_{12}^2 \\
(m_{13} - \gamma r_{23} - u_0 r_{33})^2 &= \alpha_u^2 \, r_{13}^2
\end{aligned}
$$

Sumamos las tres ecuaciones y sacamos $\alpha_u^2$ de factor común:

$$
(m_{11} - \gamma r_{21} - u_0 r_{31})^2 + (m_{12} - \gamma r_{22} - u_0 r_{32})^2 + (m_{13} - \gamma r_{23} - u_0 r_{33})^2
= \alpha_u^2 \, (r_{11}^2 + r_{12}^2 + r_{13}^2).
$$


$$
\alpha_u^2 = (m_{11} - \gamma r_{21} - u_0 r_{31})^2 + (m_{12} - \gamma r_{22} - u_0 r_{32})^2 + (m_{13} - \gamma r_{23} - u_0 r_{33})^2.
$$

El lado derecho es la suma de los cuadrados de las componentes del vector $\bar m_1 - \gamma \bar r_2 - u_0 \bar r_3$. Eso es lo mismo que hacer el producto interno de ese vector por sí mismo, así que lo volvemos a escribir en forma vectorial:

$$
\alpha_u^2 = \langle \bar m_1 - \gamma \bar r_2 - u_0 \bar r_3,\ \bar m_1 - \gamma \bar r_2 - u_0 \bar r_3 \rangle.
$$

Sacamos raíz. Como $\alpha_u > 0$, nos quedamos con la positiva:

$$
\alpha_u = \sqrt{\langle \bar m_1 - \gamma \bar r_2 - u_0 \bar r_3,\ \bar m_1 - \gamma \bar r_2 - u_0 \bar r_3 \rangle} = \|\bar m_1 - \gamma \bar r_2 - u_0 \bar r_3\|_2.
$$

Las tres ecuaciones de antes de elevar al cuadrado, en forma vectorial, son $\bar m_1 - \gamma \bar r_2 - u_0 \bar r_3 = \alpha_u \bar r_1$. Con $\alpha_u$ ya calculado, pasamos $\alpha_u$ dividiendo:

$$
\bar r_1 = \frac{\bar m_1 - \gamma \bar r_2 - u_0 \bar r_3}{\alpha_u}.
$$

Por último, de la cuarta columna, $m_{14} = \alpha_u t_1 + \gamma t_2 + u_0 t_3$, despejamos:

$$
t_1 = \frac{m_{14} - \gamma t_2 - u_0 t_3}{\alpha_u}.
$$


**Armado de $K$**

Con todo lo que despejamos:

$$
K =
\begin{bmatrix}
\alpha_u & \gamma & u_0 \\
0 & \alpha_v & v_0 \\
0 & 0 & 1
\end{bmatrix}.
$$

In [2]:
def devolverMatricesCalibracion3D(M, simple=True):
    M_monio = M / norma(M[2, 0:3])

    if simple:
        M1 = M_monio[0:3, 0:3]
        M2 = M_monio[:, 3]

        M1Inversa = inversa(M1)
        R_traspuesta, K_inversa = calculaQR(M1Inversa, 'GS')

        R = traspuesta(R_traspuesta)
        K = inversa(K_inversa)
        t = arroba(K_inversa, M2)

        return R, t, K

    else:
        R = np.zeros((3, 3))
        t = np.zeros((3, 1))
        K = np.zeros((3, 3))

        # Fila 3: m3 = r3, m34 = t3
        R[2] = M_monio[2, 0:3]
        t[2] = M_monio[2, 3]

        # Fila 2: m2 = alfa_v r2 + v0 r3
        #         m24 = alfa_v t2 + v0 t3
        v0 = producto_interno(M_monio[1, 0:3], R[2])
        aux = M_monio[1, 0:3] - v0 * R[2]
        alfa_v = np.sqrt(producto_interno(aux, aux))

        R[1] = aux / alfa_v
        t[1] = (M_monio[1, 3] - v0 * t[2]) / alfa_v

        # Fila 1: m1 = alfa_u r1 + gamma r2 + u0 r3
        #         m14 = alfa_u t1 + gamma t2 + u0 t3
        u0 = producto_interno(M_monio[0, 0:3], R[2])
        gamma = producto_interno(M_monio[0, 0:3], R[1])
        aux = M_monio[0, 0:3] - gamma * R[1] - u0 * R[2]
        alfa_u = np.sqrt(producto_interno(aux, aux))

        R[0] = aux / alfa_u
        t[0] = (M_monio[0, 3] - gamma * t[1] - u0 * t[2]) / alfa_u

        # Armamos K con los parámetros que despejamos
        K[0, 0] = alfa_u
        K[0, 1] = gamma
        K[0, 2] = u0
        K[1, 1] = alfa_v
        K[1, 2] = v0
        K[2, 2] = 1

        return R, t, K

### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

In [6]:
def obtenerQ2d(imgPts,objPts):
  matriz_Q=[] #Armo mi matriz Q en donde voy a guardar las filas creadas

  for i in range(len(objPts)): #Separo las coordenadas y aplico el algoritmo
    U= objPts[i][0]
    V= objPts[i][1]
    u= imgPts[i][0]
    v= imgPts[i][1]

    fila_1=[U, V, 1, 0, 0, 0, -u*U, -u*V, -u]
    fila_2=[0, 0, 0, U, V, 1, -v*U, -v*V, -v]

    matriz_Q.append(fila_1)
    matriz_Q.append(fila_2)

  return np.array(matriz_Q)

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---